In [2]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

In [3]:
GOOGLE_API_KEY = "***"

In [4]:
# This is the global variable to store document content
document_content = ""

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [5]:
@tool
def update(content: str) -> str:
    """Updates the document with the provided content"""

    global document_content
    document_content = content
    return f"Document has been updated. The current content is: \n{document_content}"

@tool
def save(filename: str) -> str:
    """Save the current document to a text file and finish the process.
    
    Args:
        filename: Name for the text file.
    """
    global document_content
    
    if not filename.endswith('.txt'):
        filename = f"{filename}.txt"

    try:
        with open(filename, 'w') as file:
            file.write(document_content)
        print(f"\n💾 Document has been saved to: {filename}")
        return f"Document has been saved successfully to '{filename}'."
    
    except Exception as e:
        return f"Error saving document: {str(e)}"


tools = [update, save]

In [6]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY).bind_tools(tools)

In [7]:
def our_agent(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content=f"""
    You are Drafter, a helpful writing assistant. You are going to help the user update and modify documents.
    
    - If the user wants to update or modify content, use the 'update' tool with the complete updated content.
    - If the user wants to save and finish, you need to use the 'save' tool.
    - Make sure to always show the current document state after modifications.
    
    The current document content is:{document_content}
    """)

    if not state["messages"]:
        user_input = "I'm ready to help you update a document. What would you like to create?"
        user_message = HumanMessage(content=user_input)
    else:
        user_input = input("\nWhat would you like to do with the document? ")
        print(f"\n👤 USER: {user_input}")
        user_message = HumanMessage(content=user_input)

    all_messages = [system_prompt] + list(state["messages"]) + [user_message]
    response = model.invoke(all_messages)

    print(f"\n🤖 AI: {response.content}")
    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"🔧 USING TOOLS: {[tc['name'] for tc in response.tool_calls]}")

    return {"messages": list(state["messages"]) + [user_message, response]}

In [8]:
def should_continue(state: AgentState) -> str:
    """Determine if we should continue or end the conversation."""

    messages = state["messages"]
    
    if not messages:
        return "continue"
    
    # This looks for the most recent tool message....
    for message in reversed(messages):
        # ... and checks if this is a ToolMessage resulting from save
        if (isinstance(message, ToolMessage) and 
            "saved" in message.content.lower() and
            "document" in message.content.lower()):
            return "end" # goes to the end edge which leads to the endpoint
        
    return "continue"

In [9]:
def print_messages(messages):
    """Function to print the messages in a more readable format"""
    if not messages:
        return
    
    for message in messages[-3:]:
        if isinstance(message, ToolMessage):
            print(f"\n🛠️ TOOL RESULT: {message.content}")

In [10]:
graph = StateGraph(AgentState)

graph.add_node("agent", our_agent)
graph.add_node("tools", ToolNode(tools))

graph.set_entry_point("agent")

graph.add_edge("agent", "tools")


graph.add_conditional_edges(
    "tools",
    should_continue,
    {
        "continue": "agent",
        "end": END,
    },
)

app = graph.compile()

In [11]:
def run_document_agent():
    print("\n ===== DRAFTER =====")
    
    state = {"messages": []}
    
    for step in app.stream(state, stream_mode="values"):
        if "messages" in step:
            print_messages(step["messages"])
    
    print("\n ===== DRAFTER FINISHED =====")

In [12]:
if __name__ == "__main__":
    run_document_agent()


 ===== DRAFTER =====

🤖 AI: [{'type': 'text', 'text': 'What is the initial content for the document?', 'extras': {'signature': 'Cn0BcsjafCFrRA7UX+MpH85lnQHd/+AEtLh85DLlD8lxdOo7g5dWqIhpAdMKFcOcFfDie2DCOBKXnHYWN8fz1JozdZtmnW1KM4HvqAjbS2J1NuX+8Qj//iLldJr6if94y/kRw6l5criOYPUCE4Ddb7Ys2g+neFVo9iG6joWhvA=='}}]



What would you like to do with the document?  Write me an email for my company's HR asking for a 60% raise in salary.



👤 USER: Write me an email for my company's HR asking for a 60% raise in salary.

🤖 AI: 
🔧 USING TOOLS: ['update']

🛠️ TOOL RESULT: Document has been updated. The current content is: 
Subject: Salary Review Request

Dear [HR Manager Name],

I am writing to formally request a review of my current salary and to propose an adjustment to reflect my contributions and market value. I am seeking a 60% increase in my current salary.

Since joining [Company Name] on [Start Date], I have consistently delivered strong results and taken on increasing responsibilities. [Provide specific examples of achievements, projects, and contributions that demonstrate your value to the company].

I am confident that a 60% salary increase aligns with my performance, experience, and the current market rates for similar roles. I have conducted thorough research and believe this adjustment is fair and equitable.

I am very passionate about my work at [Company Name] and committed to its continued success. I am eage


What would you like to do with the document?  Fill in the start date as July 2024 and Company Name as Valley Inc.



👤 USER: Fill in the start date as July 2024 and Company Name as Valley Inc.

🤖 AI: 
🔧 USING TOOLS: ['update']

🛠️ TOOL RESULT: Document has been updated. The current content is: 
Subject: Salary Review Request

Dear [HR Manager Name],

I am writing to formally request a review of my current salary and to propose an adjustment to reflect my contributions and market value. I am seeking a 60% increase in my current salary.

Since joining [Company Name] on [Start Date], I have consistently delivered strong results and taken on increasing responsibilities. [Provide specific examples of achievements, projects, and contributions that demonstrate your value to the company].

I am confident that a 60% salary increase aligns with my performance, experience, and the current market rates for similar roles. I have conducted thorough research and believe this adjustment is fair and equitable.

I am very passionate about my work at [Company Name] and committed to its continued success. I am eager to

...


👤 USER: Great, save it please.

🤖 AI: [{'type': 'text', 'text': 'I can save the document for you. What would you like to name the file?', 'extras': {'signature': 'CpABAXLI2nyfnnkoMLmNlbj3UN2E0aIwuRUSNMY0PWrSOaqw39cEQXer+2So/CULeuyAjJ/WQH7LmxpcmQeifZsmlE9L7/NeD0H2bvlYvMw/MFqahDOsnF9FLtVLpUCHfbUMCqwUZjnVBOHC9FmbOCM3mjLcGzz0CthaTo+Zbtbcc5hz1aP+g3fmV09rJHbeX9Ze'}}]

🛠️ TOOL RESULT: Document has been updated. The current content is: 
Subject: Salary Review Request

Dear [HR Manager Name],

I am writing to formally request a review of my current salary and to propose an adjustment to reflect my contributions and market value. I am seeking a 60% increase in my current salary.

Since joining Valley Inc. on July 2024, I have consistently delivered strong results and taken on increasing responsibilities. [Provide specific examples of achievements, projects, and contributions that demonstrate your value to the company].

I am confident that a 60% salary increase aligns with my performance, exper


What would you like to do with the document?  Save the document.



👤 USER: Save the document.

🤖 AI: [{'type': 'text', 'text': 'I can save the document for you, but I need a filename. What would you like to name the file (e.g., "Salary Review Request.txt")?', 'extras': {'signature': 'CscDAXLI2nz3f5GyHYXecqa7+m4dg1gz5CIIlgznH0pPUKELfA6dXAu+5j0HMtDhzW4F6I9EyjYaZTrNVNwCR+MXoiH4km8AlkufdcPrGwDwVpzR8+paoCRGVvw/b4E1tlc0K7bklfQqInaEOBzW8UsZUyQIxi8zgZGdFL4r3CiptnSh56mQoNxnYgHSFVBfUMBy5R2qtcGpoxHWaF/1TExf83ZJ+ic+UtOQTMuITXmacXTUUMILS93oR1hN4i6cAzNAIbGduDIBuf2Vr8kYKnCYDG4J+E4zGu5x6WP93OBd5cBmVQv6nZqxWHy7rsWqMsR8F3ZkbWlGoWEFnN/q/ZCyVz30TUyXCYeufII4DU69OXRPVCK7g90o9NX7oPDp8lY7WDIkPw3IpqsEO4YOSI41eWJ1kJcjHwVp81cCH68HNRFNHR0Xf29RT6FTRcOeZUsDnburA/F8rn7qMjs5To3KJJBu45u/D08A2epqKdj74NYbhA7aBvxmohhnK49DuogBtYSomdwyImtEvoyVEvZSwfk1tBGomEK23Oyt/j2HVTIB0K4+xZGanWfYwyfECnasacsyA616lG3++2hkC8gqxE7hWg/f114='}}]



What would you like to do with the document?  Please make up a file name and save it as txt file.



👤 USER: Please make up a file name and save it as txt file.

🤖 AI: 
🔧 USING TOOLS: ['save']

💾 Document has been saved to: Salary_Review_Request.txt

🛠️ TOOL RESULT: Document has been saved successfully to 'Salary_Review_Request.txt'.

 ===== DRAFTER FINISHED =====
